In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("Retail_Dashboard.xlsx")
df.head()

,Unnamed: 0,Unnamed: 1
0,NaN,NaN
1,Row Labels,Sum of Revenue
2,Australia,138453.81
3,Austria,10198.68
4,Bahrain,548.4


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  40 non-null     object
 1   Unnamed: 1  39 non-null     object
dtypes: object(2)
memory usage: 788.0+ bytes


In [4]:
df_raw = pd.read_excel("Retail_Dashboard.xlsx", header=None)
df_raw.head(15)

,0,1
0,NaN,NaN
1,NaN,NaN
2,Row Labels,Sum of Revenue
3,Australia,138453.81
4,Austria,10198.68
5,Bahrain,548.4
6,Belgium,41196.34
7,Brazil,1143.6
8,Canada,3666.38
9,Channel Islands,20440.54


In [5]:
df.columns

Index(['Unnamed: 0', 'Unnamed: 1'], dtype='object')

In [6]:
df.head(10)

,Unnamed: 0,Unnamed: 1
0,NaN,NaN
1,Row Labels,Sum of Revenue
2,Australia,138453.81
3,Austria,10198.68
4,Bahrain,548.4
5,Belgium,41196.34
6,Brazil,1143.6
7,Canada,3666.38
8,Channel Islands,20440.54
9,Cyprus,13502.85


In [7]:
xls = pd.ExcelFile("/content/Retail_Dashboard.xlsx")
xls.sheet_names


['Revenue_by_country',
 'Revenue_by_product',
 'Dashboard',
 'Weekly_Revenue',
 'Retail_cleaned.xlsx']

In [8]:
import pandas as pd

# Load dataset
df = pd.read_excel("/content/Retail_Dashboard.xlsx", sheet_name="Retail_cleaned.xlsx")

# Preview shape
print("Rows:", df.shape[0], " Columns:", df.shape[1])

df.head()

Rows: 1048575  Columns: 13


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Year,Month,Week,Sale_Type
0,536365.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6.0,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,Dec,49,Normal
1,536365.0,71053,WHITE METAL LANTERN,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,Dec,49,Normal
2,536365.0,84406B,CREAM CUPID HEARTS COAT HANGER,8.0,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,2010,Dec,49,Normal
3,536365.0,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,Dec,49,Normal
4,536365.0,84029E,RED WOOLLY HOTTIE WHITE HEART.,6.0,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,Dec,49,Normal


In [9]:
df = df.dropna(subset=["CustomerID"])

In [10]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [11]:
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

In [12]:
df = df[df["Quantity"] > 0]

In [13]:
df.shape

(392692, 14)

In [14]:
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Year', 'Month',
       'Week', 'Sale_Type', 'TotalAmount'],
      dtype='object')

In [15]:
import datetime as dt

# Reference date = one day after last transaction
reference_date = df["InvoiceDate"].max() + dt.timedelta(days=1)

reference_date


Timestamp('2011-12-10 12:50:00')

In [16]:
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (reference_date - x.max()).days,   # Recency
    "InvoiceNo": "nunique",                                     # Frequency
    "TotalAmount": "sum"                                        # Monetary
})

rfm.rename(columns={
    "InvoiceDate": "Recency",
    "InvoiceNo": "Frequency",
    "TotalAmount": "Monetary"
}, inplace=True)

rfm.head()


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [17]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 5, labels=[1,2,3,4,5]).astype(int)

rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

rfm.head()


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
CustomerID,,,,,,,
12346.0,326,1,77183.60,1,1,5,115
12347.0,2,7,4310.00,5,5,5,555
12348.0,75,4,1797.24,2,4,4,244
12349.0,19,1,1757.55,4,1,4,414
12350.0,310,1,334.40,1,1,2,112


In [18]:
def segment_customer(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "Recent Customers"
    elif r == 3 and f <= 2:
        return "Potential Loyalist"
    elif r == 2 and f >= 3:
        return "At Risk"
    elif r == 1 and f >= 2:
        return "Hibernating"
    else:
        return "Churn Risk"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

rfm.head(10)


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
CustomerID,,,,,,,,
12346.0,326,1,77183.60,1,1,5,115,Churn Risk
12347.0,2,7,4310.00,5,5,5,555,Champions
12348.0,75,4,1797.24,2,4,4,244,At Risk
12349.0,19,1,1757.55,4,1,4,414,Recent Customers
12350.0,310,1,334.40,1,1,2,112,Churn Risk
12352.0,36,8,2506.04,3,5,5,355,Loyal Customers
12353.0,204,1,89.00,1,1,1,111,Churn Risk
12354.0,232,1,1079.40,1,1,4,114,Churn Risk
12355.0,214,1,459.40,1,1,2,112,Churn Risk


In [19]:
rfm["Segment"].value_counts()

,count
Segment,
Loyal Customers,1003
Champions,957
Churn Risk,765
Hibernating,501
At Risk,442
Potential Loyalist,351
Recent Customers,319


In [20]:
customer_region = df.groupby("CustomerID")["Country"].first()

rfm = rfm.merge(customer_region, on="CustomerID", how="left")

rfm.head()


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Country
CustomerID,,,,,,,,,
12346.0,326,1,77183.60,1,1,5,115,Churn Risk,United Kingdom
12347.0,2,7,4310.00,5,5,5,555,Champions,Iceland
12348.0,75,4,1797.24,2,4,4,244,At Risk,Finland
12349.0,19,1,1757.55,4,1,4,414,Recent Customers,Italy
12350.0,310,1,334.40,1,1,2,112,Churn Risk,Norway


In [21]:
rfm.reset_index(inplace=True)

rfm.to_csv("/content/RFM_Output1.csv", index=False)

print("File saved successfully")


File saved successfully
